# Fork-2 runner (Colab)

**Thin runner only.** All pipeline logic lives in the git-tracked R scripts — `scripts/fork2_01_typing_floors.R` (harmonize → ONE FlowSOM cross-cohort cell-typing → Gate B canary → Gate A floors) and `scripts/fork2_02_gateC_proximity.R` (Gate C: CD8↔CK⁺ tumour proximity on **Meyer only** + type-preserving permutation control). This notebook mounts Drive, points the scripts at the Drive data paths, installs the Bioconductor IMC stack, runs them, and prints **R1–R5**. It **stops after Gate C** — METABRIC (Gate D) is untouched.

### Before you run
1. **Move the 3 data files to Drive** per the manifest:
   - `→ /content/drive/MyDrive/penumbra/data/meyer/sce_ALL_sub.rds` (84 MB)
   - `→ /content/drive/MyDrive/penumbra/data/metabric/SingleCells.fst` (326 MB)
   - `→ /content/drive/MyDrive/penumbra/data/metabric/IMCClinical.fst` (12 KB; Gate D only, stage it now)
2. **Push the latest repo commits to GitHub** (`github.com/hcl1216/penumbra`) — the notebook clones the code from there.

### Runtime / RAM
- **High-RAM runtime recommended.** Front end reads a 341 MB `.fst` + builds a ~1.21M×17 matrix (FlowSOM SOM trained on a 200k subsample then mapped to all cells → no OOM). Gate C runs on Meyer (~93k cells); its **type-preserving permutation null is the slow part (~10–20 min at PERM_N=499)** — lower `PERM_N` for a smoke test.
- Expect ~10–20 min Bioconductor install + ~10–20 min front end + ~10–20 min Gate C.

### Repo access — FLAG (confirm before running)
Couldn't verify whether `github.com/hcl1216/penumbra` is **public or private**. PUBLIC → clone cell works as-is. PRIVATE → set `GITHUB_PAT` in the clone cell. Don't commit the token.

In [ ]:
# 1) Mount Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/penumbra'
import os
for p in [f'{DRIVE}/data/meyer/sce_ALL_sub.rds',
          f'{DRIVE}/data/metabric/SingleCells.fst',
          f'{DRIVE}/data/metabric/IMCClinical.fst']:
    print(('OK   ' if os.path.exists(p) else 'MISSING '), p)
os.makedirs(f'{DRIVE}/results', exist_ok=True)

In [ ]:
# 2) Get the code from git (logic comes from the repo, NOT Drive)
GITHUB_PAT = ''   # leave '' if the repo is PUBLIC; paste a read PAT if PRIVATE
REPO_URL = 'github.com/hcl1216/penumbra.git'
url = f'https://{GITHUB_PAT + "@" if GITHUB_PAT else ""}{REPO_URL}'
import os
if os.path.isdir('/content/penumbra/.git'):
    !cd /content/penumbra && git pull --ff-only
else:
    !git clone {url} /content/penumbra
REPO = '/content/penumbra'
assert os.path.exists(f'{REPO}/scripts/fork2_01_typing_floors.R'), 'script not found — check repo/branch'
print('repo at', REPO)

In [ ]:
# 3) Point the script at the Drive data + results paths (read by the R script via env vars)
import os
os.environ['MEYER_SCE']            = f'{DRIVE}/data/meyer/sce_ALL_sub.rds'
os.environ['METABRIC_SC']          = f'{DRIVE}/data/metabric/SingleCells.fst'
os.environ['PENUMBRA_RESULTS_DIR'] = f'{DRIVE}/results'
print({k: os.environ[k] for k in ['MEYER_SCE','METABRIC_SC','PENUMBRA_RESULTS_DIR']})

In [ ]:
%%bash
# 4) System libs THEN the Bioconductor IMC stack. imcRtools hard-imports cytomapper -> EBImage ->
#    magick + fftwtools, so Magick++ / fftw3 / image libs must be present before R compiles them.
#    Slow cell (~20-40 min: EBImage/cytomapper/imcRtools compile from source). Fails loudly if missing.
apt-get -qq update
apt-get -qq install -y \
  libgdal-dev libgeos-dev libproj-dev libudunits2-dev \
  libmagick++-dev libfftw3-dev libtiff-dev libjpeg-dev libpng-dev \
  libfontconfig1-dev libfreetype6-dev libharfbuzz-dev libfribidi-dev >/dev/null
Rscript -e '
  options(repos=c(CRAN="https://cloud.r-project.org"), timeout=3600, Ncpus=parallel::detectCores())
  if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager")
  for (p in c("fst","survival")) if (!requireNamespace(p,quietly=TRUE)) install.packages(p)
  bioc <- c("SingleCellExperiment","FlowSOM","imcRtools")   # imcRtools pulls cytomapper/EBImage itself
  need <- bioc[!sapply(bioc, requireNamespace, quietly=TRUE)]
  if (length(need)) BiocManager::install(need, update=FALSE, ask=FALSE)
  ok <- sapply(c(bioc,"fst","survival"), requireNamespace, quietly=TRUE)
  print(ok)
  if (!all(ok)) stop("INSTALL FAILED: ", paste(names(ok)[!ok], collapse=", "))
  cat("ALL INSTALLED OK\n")'


In [ ]:
# 5) Run the git-tracked R script via Rscript (inherits the env vars set above; --file gives REPO)
!cd /content/penumbra && Rscript scripts/fork2_01_typing_floors.R

In [ ]:
# 6) Print R1-R4 (persisted to Drive results/) for review -> paste these back before Gate C
for r in ['fork2_R1_marker_status.md','fork2_R2_celltypes.md','fork2_R3_canary.md','fork2_R4_floors.md']:
    p = f'{DRIVE}/results/{r}'
    print('\n' + '='*78 + f'\n{r}\n' + '='*78)
    print(open(p).read() if os.path.exists(p) else '(missing — the run did not reach this artifact)')

In [ ]:
# 7) GATE C (Meyer only): CD8<->CK+ tumour proximity + type-preserving permutation control.
#    Needs results/fork2_celltypes_meyer.csv from cell 5 (front end) already on Drive.
#    Permutation is the slow part (~10-20 min at PERM_N=499). For a quick smoke test, set PERM_N lower:
# import os; os.environ['PERM_N'] = '99'
!cd /content/penumbra && Rscript scripts/fork2_02_gateC_proximity.R

In [ ]:
# 8) Print R5 (Gate C) for review -> paste back before Gate D
p = f'{DRIVE}/results/fork2_R5_gateC.md'
print(open(p).read() if os.path.exists(p) else '(missing — Gate C did not complete)')